# Let Unitree Go2 Robot Dog Trotting and Walking
Import mujoco for simulation, jax for differentiable programming, brax for environment definition and mediapy for visualization.
Feel free to use other familiar libraries for illustration.

In [351]:
import mujoco
from mujoco import mjx
import jax
import jax.numpy as jnp
import numpy as np
from jax import config  # Analytical gradients work much better with double precision.
config.update("jax_debug_nans", True)
config.update("jax_enable_x64", True)
config.update('jax_default_matmul_precision', 'high')

from brax.envs.base import Env, PipelineEnv, State
from brax.io import html, mjcf, model

import mediapy as media

Define an RL environment for quadrupedal locomotion for Unitree Go2 robot dog. The mujoco xml file can also be found from mujoco_menagerie library which hosts a zoo of robot models. Override the actuator gain to make the commanded motion better tracked. 

In [ ]:
#for mjx
class QuadrupedUnitreeGo2(PipelineEnv):
    def __init__(self, **kwargs):
        self.mj_model = mujoco.MjModel.from_xml_path('unitree_go2/scene_mjx.xml')
        self.mj_data = mujoco.MjData(self.mj_model)
        self.mjx_model = mjx.put_model(self.mj_model)
        self.mjx_data = mjx.put_data(self.mj_model, self.mj_data)

        self.init_qpos = self.mj_model.keyframe('home').qpos #exclude the first free base link, 
        #keep only 1-dof joint position can be measured by encoders
        self.base_height = self.forward_kinematics(self.init_qpos[7:])[2]

        kp = 230
        self.mj_model.actuator_gainprm[:, 0] = kp
        self.mj_model.actuator_biasprm[:, 1] = -kp
        sys = mjcf.load_model(self.mj_model)
        physics_steps_per_control_step = 5
        kwargs['n_frames'] = kwargs.get(
            'n_frames', physics_steps_per_control_step)
        kwargs['backend'] = 'mjx'

        super().__init__(sys, **kwargs)

    def forward_kinematics(self,q):
        if not q.shape == self.init_qpos.shape:
            q = jnp.concatenate([self.init_qpos[:7], q]) #add the first 7-dim free base link position and orientation
        new_mjx_data= self.mjx_data.replace(qpos=q)
        new_mjx_data= mjx.fwd_position(self.mjx_model, new_mjx_data)
        pos = new_mjx_data.site_xpos[self.mj_model.body('base').id]
        return pos

    def reset(self, rng):
        init_qpos = self.mj_model.keyframe('home').qpos
        #init_qpos[2] += 0.1
        qpos = jnp.array(init_qpos)
        qvel = jnp.zeros((self.sys.nv,))

        data = self.pipeline_init(qpos, qvel)

        obs = self._get_obs(data)

        metrics = {
            'joint_pos_reward': 0,
            'base_height_reward': 0,
            'base_orient_reward': 0,
            'base_linvel_reward': 0,
        }

        print("Initial position of the base link: ", self.forward_kinematics(qpos))
        return State(data, obs, 0, False, metrics)

    def _get_obs(self, data):
        #extract interested observations from state data
        position = data.qpos
        position = position[7:] #exclude the first free base link, keep only 1-dof joint position can be measured by encoders
        
        #IMU readings for base link
        sensor_id = self.mj_model.sensor('global_linvel').id
        sensor_adr = self.mj_model.sensor_adr[sensor_id]
        sensor_dim = self.mj_model.sensor_dim[sensor_id]
        base_linvel = data.sensordata[sensor_adr:sensor_adr+sensor_dim]    #count the index for global_linvel sensor
        
        base_rot = data.xmat[self.mj_model.body('base').id]

        return jnp.concatenate([
            position,
            base_linvel,
            base_rot.ravel(),
        ])

    def step(self, state, action):
        data = state.pipeline_state
        new_data = self.pipeline_step(data, action)
        obs = self._get_obs(new_data)
        base_id = self.mj_model.body("base").id
        #get reward/cost
        ##IMPLEMENT your own reward term here

        rotation_weight = 0.24
        vx_weight = 0.7
        vy_weight = 0.05
        control_weight = 0.01

        target_vx = 0.1
        vx = obs[12] #vx 
        vx = -jnp.square(vx - target_vx)
        vy = jnp.square(obs[13]) #vy

        base_rot = data.xmat[base_id]
        rotation_punishment = -jnp.sum(jnp.square(base_rot.ravel() - obs[15:24])) #punish the deviation from initial joint position
        
        control_cost = jnp.sum(jnp.square(action - data.qpos[7:])) #punish large control input

        reward = vx_weight * vx + rotation_weight * rotation_punishment - vy_weight*vy - control_cost*control_weight  #reward forward velocity and punish rotation deviation

        done = False
        return state.replace(pipeline_state=new_data, obs=obs, reward=reward, done=done)

    

Prepare initialization of trajectory with the cyclic motion insight. This is from a [tutorial](https://colab.research.google.com/github/google-deepmind/mujoco/blob/main/mjx/training_apg.ipynb) and originated from [Marc Raibert](https://en.wikipedia.org/wiki/Marc_Raibert)'s heuristic. 

In [353]:
duration = 4
framerate = 50

env = QuadrupedUnitreeGo2()

jit_step = jax.jit(env.step)

#make trott gait. reference from train_apg example of mjx
def cos_wave(t, step_period, scale):
    _cos_wave = -jnp.cos(((2 * jnp.pi) / step_period) * t)
    return _cos_wave * (scale / 2) + (scale / 2)


def dcos_wave(t, step_period, scale):
    """
    Derivative of the cos wave, for reference velocity
    """
    return ((scale * jnp.pi) / step_period) * jnp.sin(((2 * jnp.pi) / step_period) * t)


def make_kinematic_ref(sinusoid, step_k, scale=0.3, dt=1 / 50):
    """
    Makes trotting kinematics for the 12 leg joints.
    step_k is the number of timesteps it takes to raise and lower a given foot.
    A gait cycle is 2 * step_k * dt seconds long.
    """

    _steps = jnp.arange(step_k)
    step_period = step_k * dt
    t = _steps * dt

    wave = sinusoid(t, step_period, scale)
    # Commands for one step of an active front leg
    fleg_cmd_block = jnp.concatenate(
        [jnp.zeros((step_k, 1)),
         wave.reshape(step_k, 1),
         -2 * wave.reshape(step_k, 1)],
        axis=1
    )
    # Unitree Go2 has joints inverse to Anymal
    h_leg_cmd_bloc = 1 * fleg_cmd_block

    block1 = jnp.concatenate([
        jnp.zeros((step_k, 3)),
        fleg_cmd_block,
        h_leg_cmd_bloc,
        jnp.zeros((step_k, 3))],
        axis=1
    )

    block2 = jnp.concatenate([
        fleg_cmd_block,
        jnp.zeros((step_k, 3)),
        jnp.zeros((step_k, 3)),
        h_leg_cmd_bloc],
        axis=1
    )
    # In one step cycle, both pairs of active legs have inactive and active phases
    step_cycle = jnp.concatenate([block1, block2], axis=0)
    return step_cycle


/Users/oskarlinke/uni/sirl/SiRL/.venv/lib/python3.13/site-packages/brax/io/mjcf.py:480: UserWarning: Brax System, piplines and environments are not actively being maintained. Please see MJX for a well maintained JAX-based physics engine: https://github.com/google-deepmind/mujoco/tree/main/mjx. For a host of environments that use MJX, see: https://github.com/google-deepmind/mujoco_playground.
  warnings.warn(


Check the cyclic reference motion by directly apply it to the robot. It generates some trotting but slightly backward motion since the reference motion is treated as joint trajectories in free space without considering ground contacts.

In [354]:
kin_ref_qpos = make_kinematic_ref(cos_wave, 25, 0.3, 1./50) + jnp.array(env.mj_model.keyframe('home').qpos[7:])
kin_ref_qpos = jnp.concatenate([ kin_ref_qpos for _ in range(6)], axis=0)

state = env.reset(jax.random.PRNGKey(0))
rollout = [state.pipeline_state]
t = 0
while t < kin_ref_qpos.shape[0]:
  ctrl = jnp.array(kin_ref_qpos[t, :])
  state = jit_step(state, ctrl)
  # print(state.pipeline_state.qpos)
  rollout.append(state.pipeline_state)
  t+=1

media.show_video(env.render(rollout), fps=1.0/env.dt)

Initial position of the base link:  [0.19215678 0.142      0.0036275 ]


This browser does not support the video tag.


Wrappers to make loss function and gradient compatible with jax compilation. 

In [355]:
def _simulate_rollout(init_state, ctrl_array):

    def _scan_step(carry, input):
        new_state = jit_step(carry, input)
        return new_state, new_state
    
    state_final, state_traj = jax.lax.scan(_scan_step, init_state, ctrl_array)
    return state_traj

def rollout_loss(ctrl_array):
    init_state = env.reset(jax.random.PRNGKey(0))
    state_rollout = _simulate_rollout(init_state, ctrl_array)

    def _scan_cost(carry, input):
        tol = carry
        cost = -input.reward
        tol += cost
        return tol, cost
    
    tol_loss, cost_array = jax.lax.scan(_scan_cost, 0, state_rollout)
    return tol_loss

jit_loss_fun = jax.jit(lambda ctrl_array: rollout_loss(ctrl_array.reshape((-1, 12))))
jit_loss_grad = jax.jit(jax.grad(jit_loss_fun))



Sanity check and compile jitted functions for the first run.

In [356]:
print(jit_loss_fun(kin_ref_qpos.ravel()))
#print(len(jit_loss_grad(kin_ref_qpos.ravel())))


Initial position of the base link:  JitTracer(float64[3])
14.038619763471964


A simple gradient descent to refine the trotting motion for moving forward. Feel free to try other optimization techniques and learning rate.

In [357]:
# ## Adam (optax) with gradient clipping to refine trotting motion
# import optax

# ctrl = kin_ref_qpos.ravel()
# lr = 0.001
# n_iters = 200
# clip_norm = 10.0  # tame exploding gradients

# optimizer = optax.chain(
#     optax.clip_by_global_norm(clip_norm),
#     optax.adam(learning_rate=lr),
# )
# opt_state = optimizer.init(ctrl)

# @jax.jit
# def update_step(ctrl, opt_state):
#     loss, grad = jax.value_and_grad(jit_loss_fun)(ctrl)
#     updates, opt_state = optimizer.update(grad, opt_state, ctrl)
#     ctrl = optax.apply_updates(ctrl, updates)
#     return ctrl, opt_state, loss, grad
# best_iter = 0
# best_ctrl = ctrl
# best_loss = float('inf')
# loss_history = []
# for i in range(n_iters):
#     ctrl, opt_state, loss, grad = update_step(ctrl, opt_state)
#     loss_val = float(loss)
#     loss_history.append(loss_val)
#     if loss_val < best_loss:
#         best_loss = loss_val
#         best_ctrl = ctrl
#         best_iter = i
#     if i % 5 == 0:
#         print(f"iter {i}: loss = {loss_val:.4f}, grad_norm = {jnp.linalg.norm(grad):.4f}")

# print(f"final loss: {loss_history[-1]:.4f}, best loss: {best_loss:.4f} at iter {best_iter}")
# optimized_ctrl = best_ctrl.reshape((-1, 12))

In [ ]:

ctrl = kin_ref_qpos.ravel()
lr = 0.001
n_iters = 200
clip_norm = 10.0  # tame exploding gradients

best_iter = 0
best_ctrl = ctrl
best_loss = float('inf')
loss_history = []

for n in range(n_iters):
    loss_val = jit_loss_fun(ctrl)
    loss_history.append(loss_val)
    grad = jit_loss_grad(ctrl)
    grad_norm = jnp.linalg.norm(grad)
    if grad_norm > clip_norm:
        grad = (grad / grad_norm) * clip_norm
    ctrl = ctrl - lr * grad

    if loss_val < best_loss:
        best_loss = loss_val
        best_ctrl = ctrl
        best_iter = n
    if n % 5 == 0:
        print(f"iter {n}: loss = {loss_val:.4f}, grad_norm = {(grad_norm):.4f}")
    
print(f"final loss: {loss_history[-1]:.4f}, best loss: {best_loss:.4f} at iter {best_iter}")
optimized_ctrl = best_ctrl.reshape((-1, 12))

iter 0: loss = 14.0386, grad_norm = 40.1905
iter 5: loss = 12.7267, grad_norm = 42.8933
iter 10: loss = 11.9480, grad_norm = 77.4674
iter 15: loss = 11.5102, grad_norm = 34.5584
iter 20: loss = 10.6102, grad_norm = 35.0145
iter 25: loss = 9.5464, grad_norm = 24.9839
iter 30: loss = 8.4597, grad_norm = 30.3770
iter 35: loss = 7.6032, grad_norm = 20.1553
iter 40: loss = 7.2774, grad_norm = 27.7799
iter 45: loss = 6.9636, grad_norm = 36.5581
iter 50: loss = 7.1795, grad_norm = 28.8141
iter 55: loss = 7.1774, grad_norm = 37.1730
iter 60: loss = 6.9549, grad_norm = 32.3428
iter 65: loss = 7.0197, grad_norm = 84.3406
iter 70: loss = 7.2753, grad_norm = 66.2291
iter 75: loss = 8.3515, grad_norm = 108.9740
iter 80: loss = 8.3235, grad_norm = 180.5166
iter 85: loss = 8.5462, grad_norm = 266.3747
iter 90: loss = 8.5246, grad_norm = 173.9570
iter 95: loss = 8.8057, grad_norm = 366.1342
iter 100: loss = 8.5259, grad_norm = 314.1478
iter 105: loss = 8.4789, grad_norm = 335.2888
iter 110: loss = 8.3

In [359]:
## Rollout optimized trajectory + visualize
state = env.reset(jax.random.PRNGKey(0))
rollout = [state.pipeline_state]
for t in range(optimized_ctrl.shape[0]):
    state = jit_step(state, optimized_ctrl[t])
    rollout.append(state.pipeline_state)

media.show_video(env.render(rollout), fps=1.0/env.dt)

Initial position of the base link:  [0.19215678 0.142      0.0036275 ]


This browser does not support the video tag.
